Análise de Limiar de Decisão e Custos

## Objetivo
Comparar diferentes limiares de classificação para identificar
aquele que minimiza o custo financeiro das decisões de crédito.

Custos definidos no exercício:
- Falso negativo (inadimplente aprovado): R$ 8.000
- Falso positivo (bom pagador recusado): R$ 1.500

O limiar será escolhido utilizando exclusivamente os dados de treino,
com validação cruzada de 5 folds.

O conjunto de teste será utilizado somente para a avaliação final.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix
    )

In [2]:
CAMINHO_DADOS = Path("..") / "data" / "credito.csv"

df = pd.read_csv(CAMINHO_DADOS)

X = df.drop(columns=["inadimplente", "id_contrato"])
y = df["inadimplente"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)
print("\nDistribuição no treino:")
print(y_treino.value_counts(normalize=True))

Treino: (4800, 9)
Teste: (1200, 9)

Distribuição no treino:
inadimplente
0    0.79625
1    0.20375
Name: proportion, dtype: float64


In [ ]:

features_numericas = [
    "idade",
    "renda_mensal",
    "tempo_emprego_anos",
    "score_credito",
    "dividas_ativas",
    "prazo_meses",
    "valor_emprestimo"
]

features_categoricas = [
    "possui_imovel",
    "finalidade"
]


pipeline_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


pipeline_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


preprocessador = ColumnTransformer(transformers=[
    ("numericas", pipeline_numerico, features_numericas),
    ("categoricas", pipeline_categorico, features_categoricas)
])


pipeline = Pipeline(steps=[
    ("preprocessador", preprocessador),
    ("modelo", LogisticRegression(max_iter=1000, random_state=42))
])

print("Pipeline criado com sucesso!")

Pipeline criado com sucesso!


In [9]:


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


probabilidades_treino = cross_val_predict(
    pipeline,
    X_treino,
    y_treino,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("Probabilidades geradas:", len(probabilidades_treino))
print("\nPrimeiras 10 probabilidades:")

for i, probabilidade in enumerate(probabilidades_treino[:10], start=1):
    print(f"Cliente {i}: {probabilidade:.2%}")

Probabilidades geradas: 4800

Primeiras 10 probabilidades:
Cliente 1: 5.43%
Cliente 2: 22.71%
Cliente 3: 30.94%
Cliente 4: 16.81%
Cliente 5: 9.32%
Cliente 6: 3.83%
Cliente 7: 21.34%
Cliente 8: 0.99%
Cliente 9: 9.14%
Cliente 10: 8.08%


In [10]:

CUSTO_FN = 8000  # Inadimplente aprovado
CUSTO_FP = 1500  # Bom pagador recusado

def avaliar_limiar(y_real, probabilidades, limiar):


    y_previsto = (probabilidades >= limiar).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_real,
        y_previsto,
        labels=[0, 1]
    ).ravel()


    custo_total = (
        fp * CUSTO_FP +
        fn * CUSTO_FN
    )

    return {
        "Limiar": limiar,
        "Precisão": precision_score(
            y_real, y_previsto, zero_division=0
        ),
        "Recall": recall_score(
            y_real, y_previsto, zero_division=0
        ),
        "FP": fp,
        "FN": fn,
        "Custo Total (R$)": custo_total
    }

In [11]:
limiares = [0.20, 0.30, 0.40, 0.50]

resultados = []

for limiar in limiares:
    resultado = avaliar_limiar(
        y_treino,
        probabilidades_treino,
        limiar
    )

    resultados.append(resultado)

tabela_limiares = pd.DataFrame(resultados)

display(
    tabela_limiares.style.format({
        "Limiar": "{:.0%}",
        "Precisão": "{:.2%}",
        "Recall": "{:.2%}",
        "Custo Total (R$)": "R$ {:,.2f}"
    })
)

,Limiar,Precisão,Recall,FP,FN,Custo Total (R$)
0,20%,35.89%,68.92%,1204,304,"R$ 4,238,000.00"
1,30%,46.16%,51.02%,582,479,"R$ 4,705,000.00"
2,40%,54.46%,34.36%,281,642,"R$ 5,557,500.00"
3,50%,62.43%,21.57%,127,767,"R$ 6,326,500.00"


In [12]:

limiares_ampliados = [
    0.05, 0.10, 0.12, 0.15,
    0.16, 0.18, 0.20, 0.25,
    0.30, 0.40, 0.50
]

resultados_ampliados = [
    avaliar_limiar(
        y_treino,
        probabilidades_treino,
        limiar
    )
    for limiar in limiares_ampliados
]

tabela_ampliada = pd.DataFrame(resultados_ampliados)


tabela_ampliada = tabela_ampliada.sort_values(
    by="Custo Total (R$)"
).reset_index(drop=True)

display(
    tabela_ampliada.style.format({
        "Limiar": "{:.0%}",
        "Precisão": "{:.2%}",
        "Recall": "{:.2%}",
        "Custo Total (R$)": "R$ {:,.2f}"
    })
)

,Limiar,Precisão,Recall,FP,FN,Custo Total (R$)
0,16%,32.39%,78.53%,1603,210,"R$ 4,084,500.00"
1,15%,31.38%,80.37%,1719,192,"R$ 4,114,500.00"
2,18%,34.23%,73.93%,1389,255,"R$ 4,123,500.00"
3,12%,28.82%,86.40%,2087,133,"R$ 4,194,500.00"
4,20%,35.89%,68.92%,1204,304,"R$ 4,238,000.00"
5,10%,27.27%,90.90%,2371,89,"R$ 4,268,500.00"
6,25%,40.74%,59.82%,851,393,"R$ 4,420,500.00"
7,30%,46.16%,51.02%,582,479,"R$ 4,705,000.00"
8,5%,23.04%,97.44%,3183,25,"R$ 4,974,500.00"
9,40%,54.46%,34.36%,281,642,"R$ 5,557,500.00"


In [13]:
melhor_resultado = tabela_ampliada.iloc[0]

limiar_escolhido = float(melhor_resultado["Limiar"])

print(f"Limiar escolhido: {limiar_escolhido:.0%}")
print(
    f"Custo estimado no treino: "
    f"R$ {melhor_resultado['Custo Total (R$)']:,.2f}"
)
print(f"Precisão: {melhor_resultado['Precisão']:.2%}")
print(f"Recall: {melhor_resultado['Recall']:.2%}")

Limiar escolhido: 16%
Custo estimado no treino: R$ 4,084,500.00
Precisão: 32.39%
Recall: 78.53%


In [14]:

pipeline.fit(X_treino, y_treino)

probabilidades_teste = pipeline.predict_proba(X_teste)[:, 1]

print("Modelo treinado com sucesso!")
print("Probabilidades de teste:", len(probabilidades_teste))
print(f"Limiar escolhido no treino: {limiar_escolhido:.0%}")

Modelo treinado com sucesso!
Probabilidades de teste: 1200
Limiar escolhido no treino: 16%


In [15]:

resultado_16 = avaliar_limiar(
    y_teste,
    probabilidades_teste,
    limiar_escolhido
)


resultado_50 = avaliar_limiar(
    y_teste,
    probabilidades_teste,
    0.50
)

comparacao_final = pd.DataFrame([
    resultado_16,
    resultado_50
])

display(
    comparacao_final.style.format({
        "Limiar": "{:.0%}",
        "Precisão": "{:.2%}",
        "Recall": "{:.2%}",
        "Custo Total (R$)": "R$ {:,.2f}"
    })
)

,Limiar,Precisão,Recall,FP,FN,Custo Total (R$)
0,16%,32.97%,74.18%,368,63,"R$ 1,056,000.00"
1,50%,57.14%,21.31%,39,192,"R$ 1,594,500.00"


In [16]:
custo_16 = resultado_16["Custo Total (R$)"]
custo_50 = resultado_50["Custo Total (R$)"]

economia = custo_50 - custo_16

reducao_percentual = (
    economia / custo_50 * 100
    if custo_50 > 0 else 0
)

print(f"Custo com 16%: R$ {custo_16:,.2f}")
print(f"Custo com 50%: R$ {custo_50:,.2f}")
print(f"Diferença de custo: R$ {economia:,.2f}")
print(f"Redução percentual: {reducao_percentual:.2f}%")

Custo com 16%: R$ 1,056,000.00
Custo com 50%: R$ 1,594,500.00
Diferença de custo: R$ 538,500.00
Redução percentual: 33.77%


## Conclusão — Análise de Limiar e Custos

O objetivo desta etapa foi identificar um limiar de decisão que
minimizasse o custo financeiro estimado das classificações de crédito.

Foram utilizadas probabilidades obtidas por validação cruzada com
5 folds, exclusivamente sobre os 4.800 registros de treinamento,
evitando utilizar o conjunto de teste na seleção do limiar.

Entre os 11 limiares avaliados, o valor de 16% apresentou o menor
custo financeiro estimado no treino: R$ 4.084.500,00.

Após selecionar esse limiar, o modelo de Regressão Logística foi
treinado com todos os dados de treinamento e avaliado nos 1.200
registros reservados para teste.

### Resultados no teste

| Métrica | Limiar 50% | Limiar 16% |
|---|---:|---:|
| Precisão | 57,14% | 32,97% |
| Recall | 21,31% | 74,18% |
| Falsos positivos | 39 | 368 |
| Falsos negativos | 192 | 63 |
| Custo financeiro | R$ 1.594.500 | R$ 1.056.000 |

A utilização do limiar de 16% resultou em uma redução estimada
de R$ 538.500,00 (33,77%) no custo financeiro do conjunto de teste,
em comparação com o limiar padrão de 50%.

Essa redução ocorreu principalmente pela diminuição dos falsos
negativos, que representam clientes inadimplentes classificados
como bons pagadores.

Embora o limiar de 16% aumente a quantidade de bons pagadores
recusados, o custo associado a esses erros é inferior ao custo
de aprovar clientes inadimplentes, segundo as premissas do exercício.

Conclui-se que a escolha do limiar de decisão deve considerar
o impacto financeiro dos diferentes erros de classificação,
e não apenas métricas tradicionais como precisão e acurácia.

Os resultados são válidos para o conjunto de dados sintético
e para os custos definidos neste exercício. A utilização em
cenários reais exigiria validação adicional, análise de
calibração e avaliação de impactos sobre os clientes.